# 危机分级可行性实验（P6 前置）

> **说明**：本 notebook 只做研究性的可行性评估，不构成任何诊断，产出的模型**不进入产品**。

## 要回答的问题

规则模式在一份不同作者写的 52 句探针上只识别出 L3 0/12、L2 9/20。这个实验回答一件事：
**一个现成的中文预训练模型，配上公开数据和项目自己的用例，能不能在同一份探针上明显超过规则？**

- 明显超过：P6 感知模型值得提前做，共现规则层可以从简。
- 差不多：先做共现规则层，同时把精力放在独立评估集和标注数据上。

## 数据

| 文件（放在 `DRIVE_ROOT/raw/` 下） | 来源 | 用途 |
| :--- | :--- | :--- |
| `olivia/olivia_cases.csv` | 本地运行 `training/colab/export_olivia_data.py` 生成后上传 | `role=train`：四级头的训练与验证；`role=probe`：52 句探针，**只用于最后测试** |
| `sos_hl_1k/suicide_train_BERT.tsv`、`suicide_val_BERT.tsv` | SOS-HL-1K（微博“走饭”评论，心理学人员标注高 / 低自杀风险）。不存在时本 notebook 按固定提交号自动下载并校验 SHA-256 | 高风险头的训练与验证 |

SOS-HL-1K 的“高风险”大致对应项目的 L3（计划、工具、自伤行为），“低风险”里混有 L0–L2（如“真想去死”也标为低风险），
所以它**只训练一个二分类的高风险头**，不直接映射成四级。该仓库没有声明许可证：只用于本实验，权重不得随产品分发。

## 运行

1. 本地仓库根目录运行：`.venv/bin/python training/colab/export_olivia_data.py`，把 `build/colab_data/olivia_cases.csv` 上传到 Drive 的 `OLIVIA_crisis/raw/olivia/`。
2. Colab：运行时类型选 GPU（T4 即可），然后“运行时 → 全部运行”。约 15–30 分钟。
3. 断开后重新“全部运行”是安全的：模型已训练完成且配置与数据未变时跳过训练；探针已测过时不会重测（除非 `FORCE_RETEST = True`）。

## 产出（`DRIVE_ROOT/` 下）

| 路径 | 内容 |
| :--- | :--- |
| `models/crisis_teacher/model.pt`、`train_config.json` | 教师模型权重与训练配置（以后 P6 蒸馏可复用） |
| `reports/train_log.csv` | 每轮验证指标 |
| `configs/thresholds.json` | 只用验证集定出的阈值 |
| `reports/probe_predictions.csv` | 52 句探针逐条结果：规则等级、模型概率、模型等级、规则 + 模型合并后的实际等级 |
| `reports/probe_summary.md`、`probe_summary.json` | 汇总表：与规则在同一份探针上的对比 |

**怎么用**：把 `reports/probe_summary.md` 和 `probe_predictions.csv` 带回仓库对话里，据此决定共现规则层与 P6 的先后。
合并等级按 `olivia/core/safety/crisis.py` 的现行仲裁计算：规则 Stage 1 命中才可能是 L3；否则取规则与模型中较高者，最高 L2。
所以模型能把漏判抬到 L2（弹出热线卡），但单靠模型到不了 L3（急救号码）。

## 不要做的事

- 不要用探针结果回头调阈值、改超参再测：那样探针就不再是“没见过的句子”。阈值只用验证集定。
- 52 句太少，汇总里给出 95% 置信区间；这个实验只用于“选路线”，不能代替独立评估集（G-2）。

## 1. 环境

In [ ]:
# 安装固定版本依赖（Colab 已自带 torch）
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.44.2", "scikit-learn==1.5.1"], check=True)

In [ ]:
# 环境检查、随机种子与挂载 Drive
import csv, hashlib, html, json, math, os, random, re, time, urllib.request

import numpy as np
import pandas as pd
import torch
import transformers
from torch import nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif IN_COLAB:
    print("提示：当前没有 GPU。请在“运行时 → 更改运行时类型”中选择 GPU 后重新运行。")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## 2. 配置区（所有可变项集中于此）

In [ ]:
SMOKE_TEST = os.environ.get("OLIVIA_SMOKE_TEST") == "1"  # 本地 CPU 冒烟测试用；Colab 上保持关闭
DRIVE_ROOT = os.environ.get("OLIVIA_CRISIS_ROOT", "/content/drive/MyDrive/OLIVIA_crisis")

ENCODER = "hfl/chinese-roberta-wwm-ext"
SEED = 42
MAX_LEN = 160         # 含 [CLS]/[SEP] 的 token 上限
HEAD_TOKENS = 48      # 超长时保留开头这么多 token，其余从结尾取（与 SDD §10.1 双端保全同理）
BATCH_SIZE = 32
EPOCHS = 8
LR = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 3          # 验证指标连续这么多轮不提升就早停
HIGH_LOSS_WEIGHT = 1.0
VAL_FRACTION = 0.15   # 项目用例中划给验证集的比例（按等级分层）
OURS_FEED_HIGH_HEAD = True  # 项目用例也训练高风险头：L3 记为高，L0/L1 记为低，L2 不参与

TARGET_RECALL_GE2 = 0.98   # 验证集上 L2+L3 的召回目标（G-2 危机召回 ≥ 98%）
TARGET_RECALL_HIGH = 0.95  # 验证集上高风险头的召回目标

FORCE_RETRAIN = False
FORCE_RETEST = False

SOS_COMMIT = "78fb4d1ebfa0addb58417c7ad6e36c36650181c3"
SOS_URL = "https://raw.githubusercontent.com/HongzhiQ/SupervisedVsLLM-EfficacyEval/{commit}/data/suicide/{name}"
SOS_ENCODING = "gb18030"  # 原文件是 GBK 编码、CRLF 换行（GB18030 是 GBK 的超集）
SOS_FILES = {
    "train": ("suicide_train_BERT.tsv", "443b84b944d99ab357f3e6a584bd137c642d6e8557f89b8ef76ac8abe4727cc2"),
    "val": ("suicide_val_BERT.tsv", "15f76aae581a678adea3f43e1f0c96d6f5e79b782e0c113c440f12213343e2a1"),
}
PROBE_SIZE = 52
LEVEL_NAMES = ["L0", "L1", "L2", "L3"]

PATHS = {
    "olivia_csv": os.path.join(DRIVE_ROOT, "raw", "olivia", "olivia_cases.csv"),
    "sos_dir": os.path.join(DRIVE_ROOT, "raw", "sos_hl_1k"),
    "model_dir": os.path.join(DRIVE_ROOT, "models", "crisis_teacher"),
    "reports": os.path.join(DRIVE_ROOT, "reports"),
    "configs": os.path.join(DRIVE_ROOT, "configs"),
}
PATHS["model"] = os.path.join(PATHS["model_dir"], "model.pt")
PATHS["train_config"] = os.path.join(PATHS["model_dir"], "train_config.json")
PATHS["train_log"] = os.path.join(PATHS["reports"], "train_log.csv")
PATHS["thresholds"] = os.path.join(PATHS["configs"], "thresholds.json")
PATHS["probe_predictions"] = os.path.join(PATHS["reports"], "probe_predictions.csv")
PATHS["probe_summary_json"] = os.path.join(PATHS["reports"], "probe_summary.json")
PATHS["probe_summary_md"] = os.path.join(PATHS["reports"], "probe_summary.md")
PATHS["probe_marker"] = os.path.join(PATHS["reports"], ".probe_evaluated")

if SMOKE_TEST:
    EPOCHS, BATCH_SIZE, MAX_LEN, HEAD_TOKENS = 1, 8, 64, 16

for key in ("sos_dir", "model_dir", "reports", "configs"):
    os.makedirs(PATHS[key], exist_ok=True)
set_seed(SEED)
print("DRIVE_ROOT:", DRIVE_ROOT, "| SMOKE_TEST:", SMOKE_TEST)

## 3. 数据：下载校验、读取、划分与泄露检查

In [ ]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_sos():
    # 缺失时按固定提交号下载；已存在也校验 SHA-256，内容不符直接报错
    paths = {}
    for split, (name, expected) in SOS_FILES.items():
        path = os.path.join(PATHS["sos_dir"], name)
        if not os.path.exists(path):
            url = SOS_URL.format(commit=SOS_COMMIT, name=name)
            print("下载", url)
            urllib.request.urlretrieve(url, path + ".part")
            os.replace(path + ".part", path)
        actual = sha256_of(path)
        if actual != expected:
            raise ValueError(f"{name} 的 SHA-256 不符：{actual}（应为 {expected}）")
        paths[split] = path
    return paths


def clean_text(text):
    text = html.unescape(str(text)).replace(" ", " ")
    return re.sub(r"\s+", " ", text).strip()


def load_sos(path, split):
    # 三列无表头：label（1=高风险）、其反码、正文
    df = pd.read_csv(path, sep="\t", header=None, encoding=SOS_ENCODING, names=["label", "label_inverse", "text"],
                     dtype={"text": str}, quoting=csv.QUOTE_NONE, keep_default_na=False)
    if not ((df["label"] + df["label_inverse"]) == 1).all():
        raise ValueError(f"{path}: 第 1、2 列不是互补的 0/1 标签")
    return pd.DataFrame({"id": [f"sos_{split}_{i}" for i in range(len(df))],
                         "text": df["text"].map(clean_text), "level": -1,
                         "high": df["label"].astype(int), "source": "sos_hl_1k",
                         "split": split, "rule_level": -1})


def load_olivia(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 {path}：先在本地运行 export_olivia_data.py 并上传到这里")
    df = pd.read_csv(path, dtype={"id": str, "text": str, "role": str, "source": str},
                     keep_default_na=False)
    missing = {"id", "text", "level", "role", "source", "rule_level"} - set(df.columns)
    if missing:
        raise ValueError(f"olivia_cases.csv 缺少列：{sorted(missing)}")
    if not df["level"].isin([0, 1, 2, 3]).all() or not df["role"].isin(["train", "probe"]).all():
        raise ValueError("olivia_cases.csv 的 level 或 role 取值非法")
    if (df["role"] == "probe").sum() != PROBE_SIZE:
        raise ValueError(f"探针应为 {PROBE_SIZE} 条，实际 {(df['role'] == 'probe').sum()} 条")
    df["text"] = df["text"].map(clean_text)
    if OURS_FEED_HIGH_HEAD:
        df["high"] = df["level"].map({0: 0, 1: 0, 2: -1, 3: 1})
    else:
        df["high"] = -1
    return df


def stratified_split(df, fraction, seed):
    rng = np.random.default_rng(seed)
    val_idx = []
    for _, group in df.groupby("level"):
        n_val = max(1, round(len(group) * fraction))
        val_idx.extend(rng.choice(group.index.to_numpy(), size=n_val, replace=False))
    val = df.index.isin(val_idx)
    return df[~val].assign(split="train"), df[val].assign(split="val")


def bigrams(text):
    t = re.sub(r"\s+", "", text)
    return {t[i:i + 2] for i in range(len(t) - 1)} or {t}


def drop_probe_leaks(frame, probe_texts, threshold=0.8):
    # 训练 / 验证中与探针相同或高度相似（字二元组 Jaccard ≥ 0.8）的句子一律剔除
    probe_grams = [bigrams(t) for t in probe_texts]
    leak = frame["text"].map(
        lambda t: any(len(bigrams(t) & p) / len(bigrams(t) | p) >= threshold for p in probe_grams))
    for _, row in frame[leak].iterrows():
        print(f"  剔除疑似泄露 [{row['source']}] {row['text'][:60]}")
    return frame[~leak]


sos_paths = fetch_sos()
olivia = load_olivia(PATHS["olivia_csv"])
probe = olivia[olivia["role"] == "probe"].reset_index(drop=True)
ours_train, ours_val = stratified_split(olivia[olivia["role"] == "train"], VAL_FRACTION, SEED)
sos_train, sos_val = load_sos(sos_paths["train"], "train"), load_sos(sos_paths["val"], "val")

if SMOKE_TEST:
    ours_train, ours_val = ours_train.sample(48, random_state=SEED), ours_val.sample(16, random_state=SEED)
    sos_train, sos_val = sos_train.sample(48, random_state=SEED), sos_val.sample(16, random_state=SEED)

print("泄露检查：")
train_df = drop_probe_leaks(pd.concat([ours_train, sos_train], ignore_index=True), probe["text"])
val_df = drop_probe_leaks(pd.concat([ours_val, sos_val], ignore_index=True), probe["text"])
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("\n训练集：", train_df.groupby(["source", "level", "high"]).size().to_dict())
print("验证集：", val_df.groupby(["source", "level", "high"]).size().to_dict())
print("探针：", probe["level"].map(dict(enumerate(LEVEL_NAMES))).value_counts().sort_index().to_dict())
DATA_FINGERPRINT = hashlib.sha256(pd.concat([train_df, val_df])[["text", "level", "high", "split"]]
                                  .to_csv(index=False).encode("utf-8")).hexdigest()[:16]

## 4. 模型：共享编码器 + 四级头 + 高风险头

In [ ]:
from transformers import AutoModel, AutoTokenizer, BertConfig, BertModel, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(ENCODER)


class CrisisModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.level_head = nn.Linear(hidden, len(LEVEL_NAMES))
        self.high_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        cls = self.dropout(cls)
        return self.level_head(cls), self.high_head(cls).squeeze(-1)


def build_model():
    if SMOKE_TEST:  # 冒烟测试只验证流程：随机初始化的极小编码器，不下载预训练权重
        cfg = BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=32, num_hidden_layers=1,
                         num_attention_heads=2, intermediate_size=64, max_position_embeddings=MAX_LEN + 2)
        return CrisisModel(BertModel(cfg))
    return CrisisModel(AutoModel.from_pretrained(ENCODER))


def encode(text):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    budget = MAX_LEN - 2
    if len(ids) > budget:
        ids = ids[:HEAD_TOKENS] + ids[len(ids) - (budget - HEAD_TOKENS):]
    return [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]


def batches(frame, batch_size, shuffle=False, seed=SEED):
    order = np.arange(len(frame))
    if shuffle:
        np.random.default_rng(seed).shuffle(order)
    for start in range(0, len(order), batch_size):
        rows = frame.iloc[order[start:start + batch_size]]
        encoded = [encode(t) for t in rows["text"]]
        width = max(len(e) for e in encoded)
        ids = torch.full((len(encoded), width), tokenizer.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(encoded), width), dtype=torch.long)
        for i, e in enumerate(encoded):
            ids[i, :len(e)] = torch.tensor(e)
            mask[i, :len(e)] = 1
        yield (ids.to(DEVICE), mask.to(DEVICE),
               torch.tensor(rows["level"].to_numpy(), dtype=torch.long, device=DEVICE),
               torch.tensor(rows["high"].to_numpy(), dtype=torch.float, device=DEVICE))

## 5. 训练（按验证集早停；已训练完成且配置未变时跳过）

In [ ]:
from sklearn.metrics import f1_score, recall_score, roc_auc_score

AMP_DTYPE = None
if DEVICE.type == "cuda":
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

TRAIN_CONFIG = {
    "encoder": "smoke-tiny-random" if SMOKE_TEST else ENCODER, "seed": SEED, "max_len": MAX_LEN,
    "head_tokens": HEAD_TOKENS, "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR,
    "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO, "patience": PATIENCE,
    "high_loss_weight": HIGH_LOSS_WEIGHT, "val_fraction": VAL_FRACTION,
    "ours_feed_high_head": OURS_FEED_HIGH_HEAD, "sos_commit": SOS_COMMIT,
    "data_fingerprint": DATA_FINGERPRINT,
}
CONFIG_FINGERPRINT = hashlib.sha256(json.dumps(TRAIN_CONFIG, sort_keys=True).encode()).hexdigest()[:16]


def loss_fns(frame):
    levels = frame.loc[frame["level"] >= 0, "level"]
    counts = np.bincount(levels, minlength=len(LEVEL_NAMES)).astype(float)
    weights = np.sqrt(counts.sum() / (len(LEVEL_NAMES) * np.maximum(counts, 1)))
    highs = frame.loc[frame["high"] >= 0, "high"]
    pos_weight = (highs == 0).sum() / max((highs == 1).sum(), 1)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float, device=DEVICE))
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float, device=DEVICE))

    def loss(level_logits, high_logits, level, high):
        total = level_logits.new_zeros(())
        has_level, has_high = level >= 0, high >= 0
        if has_level.any():
            total = total + ce(level_logits[has_level].float(), level[has_level])
        if has_high.any():
            total = total + HIGH_LOSS_WEIGHT * bce(high_logits[has_high].float(), high[has_high])
        return total
    return loss


@torch.no_grad()
def predict(model, frame):
    model.eval()
    p_level, p_high = [], []
    for ids, mask, _, _ in batches(frame, BATCH_SIZE * 2):
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            level_logits, high_logits = model(ids, mask)
        p_level.append(torch.softmax(level_logits.float(), -1).cpu().numpy())
        p_high.append(torch.sigmoid(high_logits.float()).cpu().numpy())
    return np.concatenate(p_level), np.concatenate(p_high)


def val_metrics(model, frame):
    p_level, p_high = predict(model, frame)
    out = {}
    has_level = frame["level"].to_numpy() >= 0
    if has_level.any():
        y = frame["level"].to_numpy()[has_level]
        pred = p_level[has_level].argmax(1)
        out["level_balanced_acc"] = recall_score(y, pred, average="macro", zero_division=0)
        out["level_recall_ge2"] = float(((pred >= 2) & (y >= 2)).sum() / max((y >= 2).sum(), 1))
    has_high = frame["high"].to_numpy() >= 0
    if has_high.any():
        y = frame["high"].to_numpy()[has_high].astype(int)
        p = p_high[has_high]
        out["high_f1"] = f1_score(y, p >= 0.5, zero_division=0)
        out["high_auc"] = roc_auc_score(y, p) if len(set(y)) == 2 else float("nan")
    out["selection"] = (out.get("level_balanced_acc", 0) + out.get("high_f1", 0)) / 2
    return {k: float(v) for k, v in out.items()}


def can_skip_training():
    if FORCE_RETRAIN or not (os.path.exists(PATHS["model"]) and os.path.exists(PATHS["train_config"])):
        return False
    with open(PATHS["train_config"], encoding="utf-8") as f:
        saved = json.load(f)
    if not saved.get("training_complete"):
        print("已有模型的训练没有完成，重新训练。")
        return False
    if saved.get("config_fingerprint") != CONFIG_FINGERPRINT:
        print("配置或数据与已有模型不同，重新训练。")
        return False
    return True


def train():
    set_seed(SEED)
    model = build_model().to(DEVICE)
    loss = loss_fns(train_df)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * math.ceil(len(train_df) / BATCH_SIZE)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    with open(PATHS["train_config"], "w", encoding="utf-8") as f:
        json.dump({**TRAIN_CONFIG, "config_fingerprint": CONFIG_FINGERPRINT, "training_complete": False}, f, indent=2)
    best, stale, log = -1.0, 0, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        started, total = time.time(), 0.0
        for ids, mask, level, high in batches(train_df, BATCH_SIZE, shuffle=True, seed=SEED + epoch):
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                level_logits, high_logits = model(ids, mask)
            batch_loss = loss(level_logits, high_logits, level, high)
            optimizer.zero_grad()
            scaler.scale(batch_loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += batch_loss.item()
        metrics = val_metrics(model, val_df)
        log.append({"epoch": epoch, "train_loss": total, "seconds": round(time.time() - started), **metrics})
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in log[-1].items()})
        if metrics["selection"] > best:
            best, stale = metrics["selection"], 0
            torch.save({"state_dict": model.state_dict(), "config": TRAIN_CONFIG}, PATHS["model"])
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"验证指标 {PATIENCE} 轮未提升，早停。")
                break
    pd.DataFrame(log).to_csv(PATHS["train_log"], index=False)
    with open(PATHS["train_config"], "w", encoding="utf-8") as f:
        json.dump({**TRAIN_CONFIG, "config_fingerprint": CONFIG_FINGERPRINT, "training_complete": True,
                   "best_selection": best, "epochs_run": len(log)}, f, indent=2)


if can_skip_training():
    print("已有完整训练的模型且配置一致，跳过训练（需要重训时设 FORCE_RETRAIN = True）。")
else:
    train()

model = build_model().to(DEVICE)
model.load_state_dict(torch.load(PATHS["model"], map_location=DEVICE)["state_dict"])
model.eval()
print("已加载最佳模型，配置指纹", CONFIG_FINGERPRINT)

## 6. 阈值（只用验证集）

In [ ]:
def threshold_for_recall(scores, positives, target):
    # 满足“召回 ≥ target”的最大阈值：允许漏掉 floor((1-target)*正例数) 个正例
    pos = np.sort(scores[positives])
    if len(pos) == 0:
        return 0.5
    allowed_misses = int(math.floor((1 - target) * len(pos) + 1e-9))
    return float(pos[allowed_misses])


p_level_val, p_high_val = predict(model, val_df)
y_level_val, y_high_val = val_df["level"].to_numpy(), val_df["high"].to_numpy()
has_level, sos_rows = y_level_val >= 0, (val_df["source"] == "sos_hl_1k").to_numpy()
p_ge2_val = p_level_val[:, 2] + p_level_val[:, 3]

THRESHOLDS = {
    "ge2": threshold_for_recall(p_ge2_val[has_level], y_level_val[has_level] >= 2, TARGET_RECALL_GE2),
    "high": threshold_for_recall(p_high_val[sos_rows], y_high_val[sos_rows] == 1, TARGET_RECALL_HIGH),
    "config_fingerprint": CONFIG_FINGERPRINT,
}
l0 = has_level & (y_level_val == 0)
sos_low = sos_rows & (y_high_val == 0)
print(f"ge2 阈值 {THRESHOLDS['ge2']:.4f}：验证集 L0 中被抬到 ≥L2 的比例 "
      f"{(p_ge2_val[l0] >= THRESHOLDS['ge2']).mean():.2%}（{l0.sum()} 条）")
print(f"high 阈值 {THRESHOLDS['high']:.4f}：SOS 验证集低风险中被判高风险的比例 "
      f"{(p_high_val[sos_low] >= THRESHOLDS['high']).mean():.2%}（{sos_low.sum()} 条）")
with open(PATHS["thresholds"], "w", encoding="utf-8") as f:
    json.dump(THRESHOLDS, f, indent=2)

## 7. 探针测试（只运行一次）

模型等级：四级头取最大概率；`P(L2)+P(L3)` 过 ge2 阈值时至少 L2；高风险头过 high 阈值时记为 L3。
合并等级按 `crisis.py` 现行仲裁：规则 L3（Stage 1 命中）保持 L3；否则取规则与模型的较高者，最高 L2。

In [ ]:
def model_levels(p_level, p_high, th):
    level = p_level.argmax(1)
    level = np.where(p_level[:, 2] + p_level[:, 3] >= th["ge2"], np.maximum(level, 2), level)
    return np.where(p_high >= th["high"], 3, level)


def combined_levels(rule, model_level):
    return np.where(rule == 3, 3, np.minimum(np.maximum(rule, model_level), 2))


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (max(0.0, centre - half), min(1.0, centre + half))


def summarize(ideal, predicted):
    def count(mask, cond):
        return int((mask & cond).sum()), int(mask.sum())
    rows = {
        "L3 判为 L3": count(ideal == 3, predicted == 3),
        "L3 判为 ≥L2（弹热线卡）": count(ideal == 3, predicted >= 2),
        "L2 判为 ≥L2": count(ideal == 2, predicted >= 2),
        "L2+L3 判为 ≥L2": count(ideal >= 2, predicted >= 2),
        "L1 判为 L1": count(ideal == 1, predicted == 1),
        "L0 误报为 ≥L2": count(ideal == 0, predicted >= 2),
        "L0 误报为 ≥L1": count(ideal == 0, predicted >= 1),
    }
    return {name: {"hit": k, "n": n, "ci95": wilson(k, n)} for name, (k, n) in rows.items()}


fingerprint = f"{CONFIG_FINGERPRINT}:{THRESHOLDS['ge2']:.6f}:{THRESHOLDS['high']:.6f}"
if os.path.exists(PATHS["probe_marker"]) and not FORCE_RETEST:
    with open(PATHS["probe_marker"], encoding="utf-8") as f:
        print(f"探针已测过（{f.read().strip()}），不重复测试；下面显示上次的结果。需要重测时设 FORCE_RETEST = True。")
else:
    p_level_probe, p_high_probe = predict(model, probe)
    ideal, rule = probe["level"].to_numpy(), probe["rule_level"].to_numpy()
    model_level = model_levels(p_level_probe, p_high_probe, THRESHOLDS)
    combined = combined_levels(rule, model_level)
    predictions = probe[["id", "text", "level", "rule_level"]].rename(columns={"level": "ideal_level"}).assign(
        **{f"p_{name}": p_level_probe[:, i].round(4) for i, name in enumerate(LEVEL_NAMES)},
        p_high=p_high_probe.round(4), model_level=model_level, combined_level=combined)
    predictions.to_csv(PATHS["probe_predictions"], index=False, encoding="utf-8")
    summary = {"config_fingerprint": CONFIG_FINGERPRINT, "thresholds": THRESHOLDS, "variants": {
        "规则（现状）": summarize(ideal, rule),
        "模型单独（未经仲裁）": summarize(ideal, model_level),
        "规则 + 模型（按现行仲裁）": summarize(ideal, combined),
    }}
    with open(PATHS["probe_summary_json"], "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    names = list(next(iter(summary["variants"].values())))
    lines = ["# 危机分级可行性实验：52 句探针结果", "",
             f"配置指纹 `{CONFIG_FINGERPRINT}`；阈值只用验证集确定。区间为 Wilson 95% 置信区间。", "",
             "| 指标 | " + " | ".join(summary["variants"]) + " |",
             "| :--- | " + " | ".join(":---:" for _ in summary["variants"]) + " |"]
    for name in names:
        cells = []
        for variant in summary["variants"].values():
            r = variant[name]
            cells.append(f"{r['hit']}/{r['n']}（{r['ci95'][0]:.0%}–{r['ci95'][1]:.0%}）")
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    lines += ["", "## 规则漏判、规则 + 模型补上的句子", ""]
    lines += [f"- {r.id}（应为 {LEVEL_NAMES[r.ideal_level]}，规则 {LEVEL_NAMES[r.rule_level]} → "
              f"合并 {LEVEL_NAMES[r.combined_level]}）{r.text}"
              for r in predictions.itertuples() if r.ideal_level >= 2 and r.rule_level < 2 and r.combined_level >= 2]
    lines += ["", "## 规则 + 模型仍然漏判（应为 ≥L2，合并 < L2）", ""]
    lines += [f"- {r.id}（应为 {LEVEL_NAMES[r.ideal_level]}，合并 {LEVEL_NAMES[r.combined_level]}，"
              f"P(L2)+P(L3)={r.p_L2 + r.p_L3:.2f}，p_high={r.p_high:.2f}）{r.text}"
              for r in predictions.itertuples() if r.ideal_level >= 2 and r.combined_level < 2]
    lines += ["", "## 新增误报（应为 L0，规则 < L2，合并 ≥ L2）", ""]
    lines += [f"- {r.id}（合并 {LEVEL_NAMES[r.combined_level]}）{r.text}"
              for r in predictions.itertuples() if r.ideal_level == 0 and r.rule_level < 2 and r.combined_level >= 2]
    with open(PATHS["probe_summary_md"], "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    with open(PATHS["probe_marker"], "w", encoding="utf-8") as f:
        f.write(f"{fingerprint} @ {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

with open(PATHS["probe_summary_md"], encoding="utf-8") as f:
    print(f.read())

## 8. 怎么读结果

- 主要看 **“L2+L3 判为 ≥L2”** 这一行：规则现状是 9/32。规则 + 模型如果明显更高、且区间不与规则重叠，说明模型路线值得提前做。
- 这一行必须和 **L0 误报** 一起看：一个把所有句子都判成 L2 的模型也能拿到 32/32，同时 L0 误报 14/14。
- **L0 误报** 只有 14 句，只能看出有没有明显的副作用，不能证明满足 G-2 的“口癖误报 ≤ 5%”。
- “L3 判为 L3” 在合并一栏不会因模型而提高：没有 Stage 1 计划证据时，现行仲裁最高给 L2（防止误报 110/120）。
- 把 `reports/probe_summary.md` 和 `reports/probe_predictions.csv` 带回仓库对话即可。

In [ ]:
# 各产出文件是否存在及大小
for key in ("model", "train_config", "train_log", "thresholds", "probe_predictions",
            "probe_summary_json", "probe_summary_md", "probe_marker"):
    path = PATHS[key]
    size = f"{os.path.getsize(path) / 1024:.1f} KB" if os.path.exists(path) else "缺失"
    print(f"{key:20s} {size:>12s}  {path}")